# Maintenance Rehearsal (A) — Data Prep (extractive)

Builds **per-sentence binary labels** for a RoBERTa sentence scorer: for each
article, which of its sentences an extractive summary should keep.

- **input** = the article, truncated to `configs/chunking.yaml`'s `max_words`,
  encoded BERTSUM-style (`<s> sentence </s>` per sentence).
- **label** = 1 for sentences chosen by greedy ROUGE oracle extraction, 0
  otherwise.

## Why extractive rather than seq2seq

Maintenance rehearsal is *surface repetition* — it keeps the source's own
sentences. An earlier version of this notebook trained `t5-small` to generate
a summary and then snapped each generated sentence back onto the nearest
source sentence. Selecting sentences directly makes verbatim retention
structural instead of a post-hoc repair and removes the generation step
entirely.

It does not, however, make the novel n-gram ratio zero — joining
non-adjacent sentences creates n-grams spanning the gap between them. The
invariant is per-sentence (every kept sentence is verbatim), which is what
`seam_report` in `src/pipeline/extractive.py` actually checks.

## Prior work

| Choice | Source |
|---|---|
| RoBERTa as the extractive model, single extraction pass, dependent ratio | Sie, Beek, Bots, Brinkkemper & Gatt, NLLP 2024 (arXiv:2408.09777) **§3.2** — RoBERTa + one dependent-ratio pass was the best extractive configuration in their 5 extractive x 6 abstractive x 3 ratio grid |
| Sentence-level classification supervised by greedy-ROUGE oracle labels | Liu & Lapata, EMNLP 2019 (**BERTSUM / PreSumm**) — the standard way to derive extractive labels from abstractive-summary datasets |

> **A warning from the same paper worth carrying forward.** Sie et al. found
> that adding an extractive step made encoder-decoder abstractive models
> (T5, BART, Pegasus, LongT5, PegasusX) *worse*, while the decoder-only model
> (Llama3) reliably improved. Our elaborative rehearsal (B, `07`) is T5 —
> encoder-decoder — so the A→B path may be a loss and needs its own ablation.
> The final QA model is Llama-3.1-8B, decoder-only, which is the case where
> extraction helped.

In [1]:
import json
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import datasets
import numpy as np
import pandas as pd
import yaml
from datasets import load_dataset
from nltk.tokenize import sent_tokenize
from rouge_score import rouge_scorer
from transformers import AutoTokenizer

from src.pipeline.extractive import encode_sentences

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load `cnn_dailymail`

Official `train`/`validation`/`test` splits. `validation` drives checkpoint
selection in `05`; `test` is touched once, at the end, for the reported
number — using the selection split for the final metric would be circular.

In [2]:
MAX_TRAIN_EXAMPLES = 3000
MAX_VAL_EXAMPLES = 300
MAX_TEST_EXAMPLES = 300

train_raw = load_dataset("cnn_dailymail", "3.0.0", split=f"train[:{MAX_TRAIN_EXAMPLES}]")
val_raw = load_dataset("cnn_dailymail", "3.0.0", split=f"validation[:{MAX_VAL_EXAMPLES}]")
test_raw = load_dataset("cnn_dailymail", "3.0.0", split=f"test[:{MAX_TEST_EXAMPLES}]")
print(f"train: {len(train_raw)}, validation: {len(val_raw)}, test: {len(test_raw)}")

train: 3000, validation: 300, test: 300


## 2. Greedy ROUGE oracle → per-sentence labels

`cnn_dailymail` ships abstractive `highlights`, not extractive labels, so the
labels are approximated: greedily add whichever sentence most improves
combined ROUGE-1/2 F1 against the highlights, stopping when the gain falls
below `MIN_GAIN`.

Two things this notebook does in a specific order for a reason:

**Truncate first, then run the oracle.** Selecting from the untruncated
article could pick a sentence past the cut, labelling a sentence that is not
in the input — the model would be supervised toward something it cannot see.

**`MIN_GAIN` exists to stop the greedy search grabbing filler.** Without it
the search takes a near-irrelevant trailing sentence purely because it is
marginally better than nothing (a stray `"Copyright 2007 Reuters."` was
picked as a third sentence in an earlier run).

Unlike the previous version, the output is the **index set**, not the
concatenated text — that is what a sentence classifier needs.

In [3]:
CHUNK_MAX_WORDS = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))["max_words"]
MAX_ORACLE_SENTENCES = 3
MIN_GAIN = 0.01

print(f"CHUNK_MAX_WORDS = {CHUNK_MAX_WORDS} (from configs/chunking.yaml)")

_scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2"], use_stemmer=True)


def truncate_words(text: str, max_words: int) -> str:
    return " ".join(text.split()[:max_words])


def greedy_oracle_extract(sentences: list[str], reference_summary: str) -> list[int]:
    """Indices (ascending) of sentences that best approximate an extractive
    summary of `reference_summary`, picked greedily by combined ROUGE-1/2 F1."""
    selected: list[int] = []
    selected_text = ""
    prev_score = 0.0
    remaining = set(range(len(sentences)))
    for _ in range(MAX_ORACLE_SENTENCES):
        best_idx, best_score = None, prev_score
        for i in remaining:
            candidate = (selected_text + " " + sentences[i]).strip()
            scores = _scorer.score(reference_summary, candidate)
            score = (scores["rouge1"].fmeasure + scores["rouge2"].fmeasure) / 2
            if score > best_score:
                best_idx, best_score = i, score
        if best_idx is None or (best_score - prev_score) < MIN_GAIN:
            break
        selected.append(best_idx)
        selected_text = (selected_text + " " + sentences[best_idx]).strip()
        prev_score = best_score
        remaining.discard(best_idx)
    return sorted(selected)


def build_example(example: dict) -> dict | None:
    truncated = truncate_words(example["article"], CHUNK_MAX_WORDS)
    sentences = sent_tokenize(truncated)
    if len(sentences) < 2:  # nothing to select between
        return None
    oracle_idx = greedy_oracle_extract(sentences, example["highlights"])
    if not oracle_idx:
        return None
    return {
        "id": example["id"],
        "sentences": sentences,
        "labels": [1 if i in set(oracle_idx) else 0 for i in range(len(sentences))],
    }


train_examples = [e for e in (build_example(x) for x in train_raw) if e is not None]
val_examples = [e for e in (build_example(x) for x in val_raw) if e is not None]
test_examples = [e for e in (build_example(x) for x in test_raw) if e is not None]
print(f"train: {len(train_examples)} / {len(train_raw)}")
print(f"val:   {len(val_examples)} / {len(val_raw)}")
print(f"test:  {len(test_examples)} / {len(test_raw)}")

n_sents = [len(e["sentences"]) for e in train_examples]
n_pos = [sum(e["labels"]) for e in train_examples]
print(f"\nsentences per example: mean {np.mean(n_sents):.1f}, median {int(np.median(n_sents))}")
print(f"positives per example: mean {np.mean(n_pos):.2f}")
print(f"positive rate overall: {sum(n_pos) / sum(n_sents):.3f}  "
      f"(the class imbalance the loss has to cope with)")

CHUNK_MAX_WORDS = 200 (from configs/chunking.yaml)
train: 2999 / 3000
val:   300 / 300
test:  300 / 300

sentences per example: mean 10.4, median 10
positives per example: mean 2.07
positive rate overall: 0.200  (the class imbalance the loss has to cope with)


### Spot-check one example

The oracle should be picking lead sentences most of the time — news is
front-loaded — but not always, and it is worth seeing that with your own
eyes before training on 3,000 of them.

In [4]:
example = train_examples[0]
for i, (sentence, label) in enumerate(zip(example["sentences"], example["labels"])):
    mark = "KEEP" if label else "    "
    print(f"[{i:>2}] {mark} | {sentence[:96]}")

[ 0]      | LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 m
[ 1] KEEP | Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappoin
[ 2]      | "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a 
[ 3]      | "I don't think I'll be particularly extravagant.
[ 4]      | "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs."
[ 5]      | At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film
[ 6]      | Details of how he'll mark his landmark birthday are under wraps.
[ 7]      | His agent and publicist had no comment on his plans.
[ 8]      | "I'll


## 3. Encode BERTSUM-style with the RoBERTa tokenizer

Each sentence becomes `<s> sentence </s>`; the position of its opening `<s>`
is where the classifier head reads from.

RoBERTa's 512-token limit is architectural (learned position embeddings), so
sentences that would overflow are **dropped rather than truncated** — half a
sentence has no meaningful label. The count below is the check that
`max_words` is set correctly: if a meaningful share of sentences are being
dropped, chunks are too big for the backbone and `configs/chunking.yaml`
needs revisiting, because a dropped sentence can never be selected at
inference either.

In [5]:
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode_example(example: dict) -> dict:
    encoded = encode_sentences(example["sentences"], tokenizer)
    n = encoded["n_kept"]
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "cls_positions": encoded["cls_positions"],
        "labels": example["labels"][:n],
        "n_dropped": len(example["sentences"]) - n,
    }


def encode_all(examples: list[dict]) -> datasets.Dataset:
    rows = [encode_example(e) for e in examples]
    dropped = sum(r["n_dropped"] for r in rows)
    total = sum(len(r["labels"]) + r["n_dropped"] for r in rows)
    print(f"  sentences dropped for length: {dropped} / {total} ({dropped / total:.2%})")
    lost_positives = sum(
        sum(e["labels"]) - sum(r["labels"]) for e, r in zip(examples, rows)
    )
    print(f"  oracle-positive sentences lost: {lost_positives}")
    return datasets.Dataset.from_list([{k: v for k, v in r.items() if k != "n_dropped"} for r in rows])


print("train"); train_dataset = encode_all(train_examples)
print("val");   val_dataset = encode_all(val_examples)
print("test");  test_dataset = encode_all(test_examples)

lengths = sorted(len(r) for r in train_dataset["input_ids"])
pct = lambda q: lengths[int(q * (len(lengths) - 1))]  # noqa: E731
print(f"\ntoken length  50th={pct(0.50)}  90th={pct(0.90)}  99th={pct(0.99)}  max={lengths[-1]} / 512")

train
  sentences dropped for length: 0 / 31163 (0.00%)
  oracle-positive sentences lost: 0
val
  sentences dropped for length: 0 / 3050 (0.00%)
  oracle-positive sentences lost: 0
test
  sentences dropped for length: 0 / 3125 (0.00%)
  oracle-positive sentences lost: 0

token length  50th=273  90th=295  99th=319  max=355 / 512


## 4. Save

`05_rehearsal_maintenance_train.ipynb` reads these directly. The raw
sentence/label lists are kept alongside so the training notebook can show
readable selections without re-tokenizing.

In [6]:
OUT_DIR = Path("data/processed/rehearsal_maintenance")
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_dataset.save_to_disk(str(OUT_DIR / "train"))
val_dataset.save_to_disk(str(OUT_DIR / "val"))
test_dataset.save_to_disk(str(OUT_DIR / "test"))

for name, examples in [("train", train_examples), ("val", val_examples), ("test", test_examples)]:
    pd.DataFrame(
        [{"id": e["id"], "sentences": json.dumps(e["sentences"]), "labels": json.dumps(e["labels"])}
         for e in examples]
    ).to_csv(OUT_DIR / f"{name}_sentences_raw.csv", index=False, encoding="utf-8-sig")

print(f"Saved to: {OUT_DIR}")
for name, ds in [("train", train_dataset), ("val", val_dataset), ("test", test_dataset)]:
    print(f"  {name}: {len(ds)} examples")

Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 97784.52 examples/s] 

Saved to: data/processed/rehearsal_maintenance
  train: 2999 examples
  val: 300 examples
  test: 300 examples


## Summary

- Source: `cnn_dailymail` 3.0.0, official three-way split, 3,000/300/300.
- Articles truncated to `configs/chunking.yaml`'s `max_words` **before** the
  oracle runs, so every labelled sentence is one the model can actually see.
- Labels are approximated by greedy ROUGE-1/2 oracle extraction
  (`MAX_ORACLE_SENTENCES=3`, `MIN_GAIN=0.01`) — `cnn_dailymail` has no
  ground-truth extractive labels.
- Encoding is BERTSUM-style for `roberta-base`, one `<s>` marker per sentence.
  Watch the drop rate in §3: it should be near zero, and if it is not,
  `max_words` is too large for a 512-token backbone.
- Next: `05_rehearsal_maintenance_train.ipynb` trains the sentence scorer.
  Its manipulation check is trivially satisfied (extraction cannot produce
  novel n-grams), so the metric that matters there is whether the selected
  sentences match the oracle's.